# 06 · Conectando ao Spark Modo Cluster via Spark Connect 

**Teoria**: docs/05-pyspark-na-pratica.md

**Pré-requisito**: `make spark` (Spark Cluster: 1 master + 2 workers + um servidor Spark Connect, tudo em Docker).

🎯 **Objetivo**: conectar-se a um cluster Spark remoto usando o protocolo gRPC do Spark Connect. 

Seu processo Python aqui é um **cliente gRPC leve** — sem JVM, sem classpath do Hadoop, nada pesado.

Todo o processamento pesado acontece dentro dos containers docker (workers). 
O plano lógico não resolvido via gRPC e recebe os resultados de volta. Isso também significa: 
**os caminhos de arquivos que você referencia devem existir dentro dos 
containers**, não no seu laptop — é exatamente por isso que o `docker-compose.yml` 
faz bind-mount de `./data` em `/data` em cada serviço Spark.

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("python-app")
    .remote("sc://localhost:15002")   # Endpoint gRPC do servidor Spark Connect
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
print("🖥️  Master UI:    http://localhost:8080")
print("🖥️  Worker UIs:   http://localhost:8081  http://localhost:8082")
print("📊 Spark App UI:  http://localhost:4040")

🖥️  Master UI:    http://localhost:8080
🖥️  Worker UIs:   http://localhost:8081  http://localhost:8082
📊 Spark App UI:  http://localhost:4040


## Comprovando que a execução realmente acontece no cluster

🎯 **Objetivo**: verificar que o processamento está rodando no cluster Docker, não localmente.


## Lendo o dataset compartilhado

📌 Lembrete: `/data/...` é o caminho **dentro dos containers** (veja o 
mount de volume `x-spark-common` do `docker-compose.yml`).

Execute `make generate-data SCALE=large` no host primeiro — como `./data` está montado 
por bind, os containers o veem imediatamente, sem necessidade de etapa de cópia.


In [2]:
# Lê os dados Parquet — os arquivos estão no bind-mount /data dentro dos containeres workers
# Caminho ABSOLUTO: o volume `./data:/data` do docker-compose monta na raiz dos containers workers
sdf_vendas = spark.read.parquet("/data/bronze/vendas")

# O count() força a leitura distribuída — cada executor lê uma parte dos arquivos Parquet
print(f"vendas: {sdf_vendas.count():,} rows")
sdf_vendas.show(5)

vendas: 10,000,000 rows
+--------+--------------+----------+------+---+----+---+
|id_venda|id_funcionario|id_empresa| valor|dia| ano|mes|
+--------+--------------+----------+------+---+----+---+
|      77|          9712|        26| 12.68| 19|2026|  7|
|     103|          1629|        17| 73.29| 19|2026|  7|
|     111|          7384|         8|116.16|  9|2026|  7|
|     128|         14858|        15| 32.52|  5|2026|  7|
|     158|          3337|        39| 56.53| 26|2026|  7|
+--------+--------------+----------+------+---+----+---+
only showing top 5 rows



📌 **Observação**:

O Spark Connect retornou os dados normalmente, como se fosse uma SparkSession local. 
A diferença? Toda a computação (leitura, descompressão, contagem) ocorreu nos workers 
do cluster — seu processo Python apenas recebeu os resultados.

💡 **Dica**: abra a aba **Stages** da Spark UI para ver as Tasks distribuídas entre 
os 2 workers do cluster.

## Tour pelas Partitions

🧠 **Por quê?** — o cliente Python do Spark Connect intencionalmente **não** expõe a API 
RDD de baixo nível (`df.rdd`) — é um cliente apenas de DataFrame/SQL. Para inspecionar 
as partitions, use `spark_partition_id()` como uma coluna comum.

📌 Cada linha pertencerá a uma partition diferente. A contagem por `partition_id` revela 
como os dados estão distribuídos entre os workers.

In [3]:
from pyspark.sql.functions import spark_partition_id

# Agrupa por partition_id para ver quantas linhas cada partição contém
# spark_partition_id() é uma função que retorna o ID da partição de cada linha
partition_counts = (
    sdf_vendas.groupBy(spark_partition_id().alias("partition_id"))
    .count()
    .orderBy("partition_id")   # Ordena para facilitar a leitura
)
partition_counts.show(20)
print(f"Total partitions: {partition_counts.count()}")

+------------+-------+
|partition_id|  count|
+------------+-------+
|           0|2667874|
|           1|2792602|
|           2|2457614|
|           3|2081910|
+------------+-------+

Total partitions: 4


📌 **Análise da distribuição**:

Observe se as partições têm tamanhos equilibrados (ideal) ou se há skew (algumas 
partições com muito mais linhas que outras). Dados desbalanceados podem causar 
Tasks lentas que atrasam o Job inteiro (problema de *data skew*).

💡 **Dica**: se houver skew, considere reparticionar com `.repartition(n, col)` ou 
habilitar AQE (Adaptive Query Execution) para balanceamento automático.

In [4]:
sdf = (
    sdf_vendas
    .repartition(8)
    .groupBy(spark_partition_id().alias("partition_id"))
    .count()
    .orderBy("partition_id") 
)
sdf.show()

+------------+-------+
|partition_id|  count|
+------------+-------+
|           0|1250000|
|           1|1250000|
|           2|1250002|
|           3|1250002|
|           4|1249999|
|           5|1249999|
|           6|1249999|
|           7|1249999|
+------------+-------+



In [5]:
# Encerra a SparkSession — a conexão gRPC com o servidor Spark Connect é fechada
# O cluster Docker continua rodando (make up-cluster ainda ativo)
spark.stop()